## Script for histograms creation

In [1]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib
import yaml


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT(90)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

# Enrichment for control samples
channTitle["hasKsPM"] = "Event includes K_{S}#rightarrow#pi^{+}#pi^{-}"
channTitle["!hasKsPM"] = "Event does not include K_{S}#rightarrow#pi^{+}#pi^{-}"

channColor["hasKsPM"] = ROOT.kRed
channColor["!hasKsPM"] = ROOT.kBlue

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [2]:
base_path = "/workspace/root_notebooks/results"

################ SETTINGS ##################################
analysis_dir = "ks_pm_control_sample"
img_dir = "img"
analysis_hash = "c97f42f6a601db1c64b593e259e04dec"
root_file_name = "tag_kl_000_chann_div_1D_variables.root"
mode = "full"
nocut_hash = "09acde96"
log_flag = False
draw_data_mc_sum = True

if analysis_dir == "kl_pm_control_sample":
  delta_t_vars = ["Delta_t_fit",
                  "Delta_t_rec"]
  signal_channel = "pi+pi-pi+pi-"
elif analysis_dir == "ks_pm_control_sample":
  delta_t_vars = ["Delta_t_fit",
                  "Delta_t_tri_fit",
                  "Delta_t_rec"]
  signal_channel = "3pi0"

branching_ratio_mode = "ks_pm"
branching_ratio = 1.0

if branching_ratio_mode == "ks_pm":
    branching_ratio = 0.6920
elif branching_ratio_mode == "ks_00":
    branching_ratio = 0.3069

division = "channels"

if division == "tag":
    if analysis_dir == "kl_pm_control_sample":
        mc_flags = {"hasKsPM": "hasKsPM", "!hasKsPM": "!hasKsPM", "MC sum": "MC sum", "Data": "Data"}
    elif analysis_dir == "ks_pm_control_sample":
        mc_flags = {"has000": "has000", "!has000": "!has000", "MC sum": "MC sum", "Data": "Data"}
else:
    mc_flags = channName
############################################################

hist_file_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{root_file_name}"

#################### ANALYSIS REGISTRY #####################
import pandas as pd
analysis_registry = f"{base_path}/{analysis_dir}/{analysis_hash}/analysis_registry.csv"
df = pd.read_csv(analysis_registry)

hash_available = df["hash"]

########################## LOAD HISTOGRAM CONFIGURATION ##############################
hist_config_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_1D.yaml"
with open(hist_config_file_path, "r") as f:
    hist_config = yaml.safe_load(f)
######################################################################################

## Making histograms

In [116]:
try:
  variables_1D = list(hist_config.get("histograms_1d", {}).keys())
  TRUTH_ONLY_VARS_1D = hist_config.get("truth_only_vars_1d", [])
  TRUTH_MCTRUTH_ID_1D = hist_config.get("truth_mctruth_id_1d", 1)

  variables_1D_titles = []
  unit_1D = []
  range_X_1D = {}
  modes = {}

  for var in variables_1D:
      variables_1D_titles.append(hist_config["histograms_1d"][var].get("title", ""))
      unit_1D.append(hist_config["histograms_1d"][var].get("unit", ""))
      modes[var] = list(hist_config["histograms_1d"][var].get("range_x", {}).keys())

      range_X_1D[var] = {}
      for mode_tmp in modes[var]:
          range_X_1D[var][mode_tmp] = hist_config["histograms_1d"][var].get("range_x", {}).get(mode_tmp, [])

  print(variables_1D_titles)
except Exception as e:
  print(f"Error occurred: {e}")

['m^{inv}_{4#gamma} - m_{K^{0}}', 'm^{inv}_{6#gamma} - m_{K^{0}}', 'm^{inv}_{+-} - m_{K^{0}}', '#chi^{2}_{+-00}', '#chi^{2}_{00}', '#chi^{2}_{+-}', 'm^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}}', 'm^{inv,fit,+-00}_{#gamma#gamma,2} - m_{#pi^{0}}', 'm^{inv,fit,00}_{#gamma#gamma,1} - m_{#pi^{0}}', 'm^{inv,fit,00}_{#gamma#gamma,2} - m_{#pi^{0}}', 'm^{inv,fit,+-}_{#gamma#gamma,1} - m_{#pi^{0}}', 'm^{inv,fit,+-}_{#gamma#gamma,2} - m_{#pi^{0}}', 'm^{inv,rec}_{#gamma#gamma,1} - m_{#pi^{0}}', 'm^{inv,rec}_{#gamma#gamma,2} - m_{#pi^{0}}', '#Deltat^{fit}', '#Deltat^{tri,fit}', '#Deltat^{rec}', 'Total number of clusters', '#Delta_{6}', 'Number of neutral clusters', 'R_{+-,IP}', 'R_{000,IP}']


In [117]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

#################### OBJECT TO FIND ################################
histograms = {}

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        continue
    
    histograms[hash] = {}

    dir = hist_file.Get(hash)

    for i, var in enumerate(variables_1D):
      dir_obj = hist_file.Get(f"{hash}/{var}")

      if not isinstance(dir_obj, ROOT.TDirectory):
          continue

      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for chann in mc_flags.values():
        hist_name = f"{var}_{chann}_{mode}"

        if hist_name not in [key.GetName() for key in dir_obj.GetListOfKeys()]:
            continue
        histograms[hash][var][chann] = dir_obj.Get(hist_name).Clone()

hist_file.Close()


In [118]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

################## DRAW USING THSTACK ###########################
canvases = []
legends = []
stacks = []

for hash in histograms:    
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)

    root_file_name = f"{path_hash}/histograms.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")


    for i, var in enumerate(histograms[hash]):
        path_var = f"{path_hash}/{var}"
        os.makedirs(path_var, exist_ok=True)

        root_file.mkdir(var)
        root_file.cd(var)
        
        legend = ROOT.TLegend(0.15, 0.75, 0.84875, 0.9)
        legend.SetTextFont(62)
        legend.SetTextSize(0.025)
        legend.SetMargin(0.1)
        legend.SetNColumns(2)
        
        # Dodajemy hash do nazwy, aby uniknąć konfliktów w pamięci ROOTa przy wielu folderach
        stack = ROOT.THStack(f"{var}_stack_{hash}", "")
        
        for flag in histograms[hash][var]:
            hist = histograms[hash][var][flag]

            if not draw_data_mc_sum and (flag == "Data" or flag == "MC sum"):
                continue

            hist.SetMaximum(hist.GetBinContent(hist.GetMaximumBin()) * 1.3)

            if hist:
                hist.SetLineColor(channColor[flag])

                if log_flag:
                    hist.SetMinimum(0.1)
                
                if flag != "Data":
                    # Opcję rysowania przekazujemy bezpośrednio w metodzie Add dla THStack
                    stack.Add(hist, "hist")
                    legend.AddEntry(hist, channTitle[flag], "l")
                else:
                    hist.SetMarkerStyle(20)
                    hist.SetMarkerColor(channColor[flag])
                    stack.Add(hist, "pe1")
                    legend.AddEntry(hist, channTitle[flag], "pe1")

                hist.Write(f"{var}_{flag}_{mode}", ROOT.TObject.kOverwrite)
                            
        canvas = ROOT.TCanvas(f"{var}_canvas_{hash}", f"{var} Canvas", 600, 600)
        
        stack.Draw("NOSTACK")
        
        if stack.GetHistogram():
            resolution = stack.GetHistogram().GetXaxis().GetBinWidth(1)
            
            stack.GetHistogram().GetXaxis().SetTitle(f"{variables_1D_titles[variables_1D.index(var)]} [{unit_1D[variables_1D.index(var)]}]")
            stack.GetHistogram().GetYaxis().SetTitle(f"Counts / {resolution:.2f} {unit_1D[variables_1D.index(var)]}")

        ####### Set log y ########
        if log_flag:
          stack.SetMinimum(0.1)
          stack.SetMaximum(stack.GetMaximum() * 50)
        else:
          stack.SetMinimum(0)
                
        canvas.SetLogy(log_flag)
        ##########################

        legend.Draw()

        canvas.Draw()
        canvas.SaveAs(f"{path_var}/{var}_{mode}_logy_{log_flag}.pdf")
        canvas.Write(f"{var}_{mode}_logy_{log_flag}", ROOT.TObject.kOverwrite)
        canvas.Close()
    root_file.Close()

Error in <TFile::mkdir>: An object with name Knerec_5 exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/Knerec_5/Knerec_5_full_logy_False.pdf has been created
Error in <TFile::mkdir>: An object with name KnerecSix_5 exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/KnerecSix_5/KnerecSix_5_full_logy_False.pdf has been created
Error in <TFile::mkdir>: An object with name Kchrec_5 exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/Kchrec_5/Kchrec_5_full_logy_False.pdf has been created
Error in <TFile::mkdir>: An object with name Chi2SignalKinFit exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/Chi2Si

## Check of purity in the function of $\Delta t$

In [119]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

signal = signal_channel
sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[hash][var][sum])


######################### DIVIDE #############################
delta_t_divided = {}

for hash in delta_t_signal:
    delta_t_divided[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Divide(sum_hist)
        delta_t_divided[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided:
    path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
    os.makedirs(path_hash, exist_ok=True)

    root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")

    root_file.mkdir("purity")
    for hist, title in zip(delta_t_divided[hash], delta_t_vars):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Purity")
        hist.Draw("PE1")
        root_file.cd("purity")
        hist.Write(f"purity_{title.replace(' ', '_')}_per_channel", ROOT.TObject.kOverwrite)
        c.Draw()
        title = f"purity_{title.replace(' ', '_')}_per_channel.pdf"
        c.SaveAs(f"{path_hash}/{title}")
        c.Write(title, ROOT.TObject.kOverwrite)
        c.Close()
    root_file.Close()

        

Error in <TFile::mkdir>: An object with name purity exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/purity_Delta_t_fit_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/purity_Delta_t_tri_fit_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/purity_Delta_t_rec_per_channel.pdf has been created
Error in <TFile::mkdir>: An object with name purity exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/519c0397/efficiency_purity_plots/purity_Delta_t_fit_per_channel.pdf has been created
Info in <TCanvas::Pr

## Check of efficiency in the function of $\Delta t$

In [120]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

mode_eff = "subtract_bkg"

if mode_eff == "subtract_bkg":
  signal = signal_channel
  sum = signal_channel
else:
  signal = "MC sum"
  sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[nocut_hash][var][sum])


######################### DIVIDE #############################
delta_t_divided_signal = {}

for hash in delta_t_signal:
    delta_t_divided_signal[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        # sum_hist.Scale(branching_ratio)
        hist_temp.Divide(sum_hist)
        delta_t_divided_signal[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided_signal:
    path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
    os.makedirs(path_hash, exist_ok=True)
    root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")

    root_file.mkdir("efficiency_MC")
    root_file.cd("efficiency_MC")
    for hist, title in zip(delta_t_divided_signal[hash], delta_t_vars):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Efficiency")
        hist.Draw("PE1")
        if mode_eff == "subtract_bkg":
          hist.Write(f"efficiency_{title.replace(' ', '_')}_MC_subtract_bkg", ROOT.TObject.kOverwrite)
        else:
          hist.Write(f"efficiency_{title.replace(' ', '_')}_MC", ROOT.TObject.kOverwrite)
        c.Draw()

        if mode_eff == "subtract_bkg":
          title = f"efficiency_{title.replace(' ', '_')}_MC_subtract_bkg.pdf"
        else:
          title = f"efficiency_{title.replace(' ', '_')}_MC.pdf"
        c.SaveAs(f"{path_hash}/{title}")
        c.Write(title, ROOT.TObject.kOverwrite)
        c.Close()
    root_file.Close()
  

Error in <TFile::mkdir>: An object with name efficiency_MC exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/efficiency_Delta_t_fit_MC_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/efficiency_Delta_t_tri_fit_MC_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/efficiency_Delta_t_rec_MC_subtract_bkg.pdf has been created
Error in <TFile::mkdir>: An object with name efficiency_MC exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/519c0397/efficiency_purity_plots/efficiency_Delta_t_fit_MC_subtract

In [121]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

if mode_eff == "subtract_bkg":
  signal = "Data"
  signal_MC = signal_channel
  sum = "MC sum"
else:
  signal = "Data"
  signal_MC = ""
  sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        hist_data_tmp = histograms[hash][var][signal].Clone(f"{hash}_{var}_{signal}")
        hist_data_nocut_tmp = histograms[nocut_hash][var][signal].Clone(f"{nocut_hash}_{var}_{signal}")

        if mode_eff == "subtract_bkg":
          for key in histograms[hash][var]:
              if key == signal or key == signal_MC or key == sum:
                continue

              hist_data_tmp.Add(histograms[hash][var][key], -1.0)
              hist_data_nocut_tmp.Add(histograms[nocut_hash][var][key], -1.0)
              
        delta_t_signal[hash].append(hist_data_tmp)
        delta_t_sum[hash].append(hist_data_nocut_tmp)


######################### DIVIDE #############################
delta_t_divided_data = {}

for hash in delta_t_signal:
    delta_t_divided_data[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Sumw2()
        hist_temp.Divide(sum_hist)
        delta_t_divided_data[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided_data:
    path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
    os.makedirs(path_hash, exist_ok=True)

    root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")
    root_file.mkdir("efficiency_data")
    root_file.cd("efficiency_data")
    for hist, title in zip(delta_t_divided_data[hash], delta_t_vars):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Efficiency")
        hist.Draw("PE1")
        if mode_eff == "subtract_bkg":
          hist.Write(f"efficiency_{title.replace(' ', '_')}_data_subtract_bkg", ROOT.TObject.kOverwrite)
        else:
          hist.Write(f"efficiency_{title.replace(' ', '_')}_data", ROOT.TObject.kOverwrite)
        c.Draw()
        if mode_eff == "subtract_bkg":
            title = f"efficiency_{title.replace(' ', '_')}_data_subtract_bkg.pdf"
        else:
            title = f"efficiency_{title.replace(' ', '_')}_data.pdf"
        c.SaveAs(f"{path_hash}/{title}")
        c.Write(title, ROOT.TObject.kOverwrite)
        c.Close()
    root_file.Close()

Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure al

## Calc correction factor for $K_{S}\to\pi^+\pi^-$

In [122]:
corr_factor = {}

status = False
if delta_t_divided_data and delta_t_divided_signal:
    for hash in delta_t_divided_data:
        corr_factor[hash] = []
        for hist_data, hist_tag in zip(delta_t_divided_data[hash], delta_t_divided_signal[hash]):
            hist_temp = hist_data.Clone()
            hist_temp.Divide(hist_tag)
            corr_factor[hash].append(hist_temp)

    for hash in corr_factor:
        path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
        os.makedirs(path_hash, exist_ok=True)
        root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
        root_file = ROOT.TFile(root_file_name, "UPDATE")
        root_file.mkdir("corr_factor")
        root_file.cd("corr_factor")
        for hist, title in zip(corr_factor[hash], delta_t_vars):
            c = ROOT.TCanvas("")
            hist.SetMarkerStyle(ROOT.kFullCircle)
            hist.SetLineColor(ROOT.kBlack)
            hist.GetYaxis().SetRangeUser(0, 1.1)
            hist.GetYaxis().SetTitle("Correction Factor")
            hist.Draw("PE1")
            if mode_eff == "subtract_bkg":
                hist.Write(f"corr_factor_{title.replace(' ', '_')}_subtract_bkg", ROOT.TObject.kOverwrite)
            else:
                hist.Write(f"corr_factor_{title.replace(' ', '_')}", ROOT.TObject.kOverwrite)
            c.Draw()
            if mode_eff == "subtract_bkg":
              title = f"corr_factor_{title.replace(' ', '_')}_subtract_bkg.pdf"
            else:
              title = f"corr_factor_{title.replace(' ', '_')}.pdf"
            c.SaveAs(f"{path_hash}/{title}")
            c.Write(title, ROOT.TObject.kOverwrite)
            c.Close()
        root_file.Close()

Error in <TFile::mkdir>: An object with name corr_factor exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/corr_factor_Delta_t_fit_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/corr_factor_Delta_t_tri_fit_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/nocuts/efficiency_purity_plots/corr_factor_Delta_t_rec_subtract_bkg.pdf has been created
Error in <TFile::mkdir>: An object with name corr_factor exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/519c0397/efficiency_purity_plots/corr_factor_Delta_t_fit_subtract_bkg.pdf has

## Calculating events, efficiencies and purities

In [87]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)

#################### OBJECT TO FIND ################################
event_counts = {}
cuts_list = {}
efficiencies = {}
purities = {}

#################### EXTRACT EVENT COUNTS #######################
for hash in hash_available:
    obj = hist_file.Get(f"{hash}/ChannelIntegrals")
    obj_cut_list = hist_file.Get(f"{hash}/CutList")

    if obj:
      chann_integrals_str = obj.GetTitle()
      event_counts[hash] = json.loads(chann_integrals_str)

    if obj_cut_list:
      raw_title = obj_cut_list.GetTitle()
      cleaned_str = raw_title.strip("[] ").replace('"', '')
      cuts_list[hash] = [c.strip() for c in cleaned_str.split(',') if c.strip()]

hist_file.Close()
#################################################################

#################### CALCULATE EFFICIENCIES AND PURITIES #######################
for hash in hash_available:
  if hash not in event_counts:
     continue

  efficiencies[hash] = {}
  purities[hash] = {}

  mc_sum = int(event_counts[hash].get("MC sum", 0))

  for flag in mc_flags.values():
      total_events = int(event_counts.get(nocut_hash, {}).get(flag, 0))
      passed_events = int(event_counts[hash].get(flag, 0))
      efficiencies[hash][flag] = passed_events / total_events if total_events > 0 else 0
      purities[hash][flag] = passed_events / mc_sum if mc_sum > 0 else 0
###############################################################################

################## PRINT RESULTS ##################
for hash in hash_available:
    if hash not in efficiencies:
        continue
    bkg = 0
    print(f"Results for cut list: {cuts_list.get(hash, [])}")
    for flag in mc_flags.values():
        event_count = int(event_counts.get(hash, {}).get(flag, 0))
        mc_sum = int(event_counts.get(hash, {}).get("MC sum", 0))
        eff = efficiencies[hash].get(flag, 0)
        pur = purities[hash].get(flag, 0)

        if flag != signal_channel and flag != "MC sum" and flag != "Data":
            bkg += event_count
        print(f"  Flag: {flag}, Event Count: {event_count / 10**6:.2f} x 10^6, Efficiency: {eff * 100:.2f}%, Purity: {pur * 100:.2f}%, (MC - Data) / Data: {100*(mc_sum - event_count) / event_count if event_count > 0 else 0:.2f}%")
    print(f"  Background (sum of non-signal, non-MC, non-Data flags): {bkg / 10**6:.2f} x 10^6")
    print()
    




Results for cut list: []
  Flag: Data, Event Count: 230.93 x 10^6, Efficiency: 456.03%, Purity: 96.23%, (MC - Data) / Data: 3.92%
  Flag: Signal, Event Count: 0.49 x 10^6, Efficiency: 13312.87%, Purity: 0.20%, (MC - Data) / Data: 49151.45%
  Flag: Regeneration, Event Count: 1.38 x 10^6, Efficiency: 15030.75%, Purity: 0.57%, (MC - Data) / Data: 17297.64%
  Flag: Omega, Event Count: 3.74 x 10^6, Efficiency: 7203.23%, Purity: 1.56%, (MC - Data) / Data: 6311.11%
  Flag: 3pi0, Event Count: 85.38 x 10^6, Efficiency: 184.50%, Purity: 35.58%, (MC - Data) / Data: 181.06%
  Flag: Semileptonic, Event Count: 60.29 x 10^6, Efficiency: 36153.02%, Purity: 25.12%, (MC - Data) / Data: 298.02%
  Flag: Other, Event Count: 88.68 x 10^6, Efficiency: 1603.80%, Purity: 36.95%, (MC - Data) / Data: 170.60%
  Flag: pi+pi-pi+pi-, Event Count: 0.01 x 10^6, Efficiency: 10494.05%, Purity: 0.00%, (MC - Data) / Data: 2722284.40%
  Flag: MC sum, Event Count: 239.98 x 10^6, Efficiency: 461.15%, Purity: 100.00%, (MC - D

### Check if efficiencies / corr factors based on mc truth and tag are similar

In [12]:
import ROOT

cut_hash= "03e630a8"

metric = "corr_factor"
var = "Delta_t_tri_fit"
ev_type = "MC"
histos = []
hist_div = None
hist_color = [ROOT.kRed, ROOT.kBlue]
leg_titles = ["Based on tag", "Based on MC truth"]
y_ranges = []
leg_position = ()
hist_title = ""

if metric == "efficiency":
  dir_name = f"{metric}_{ev_type}"
  hist_name = f"{metric}_{var}_{ev_type}"
  y_ranges.append((0.0, 1.2))
  y_ranges.append((0.0, 1.2))
  leg_position = (0.54,0.70,0.85,0.9)
  hist_title = ev_type
else:
  dir_name = f"{metric}"
  hist_name = f"{metric}_{var}"
  y_ranges.append((0.0, 1.2))
  y_ranges.append((0.95, 1.05))
  leg_position = (0.15,0.02,0.46,0.22)

eff_histograms = f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/eff_pur_corr_factor.root"

root_file = ROOT.TFile.Open(eff_histograms)

eff_dir = root_file.Get(dir_name)
histos.append(eff_dir.Get(hist_name))
histos.append(eff_dir.Get(f"{hist_name}_subtract_bkg"))

hist_div = histos[0].Clone("hist_div")
hist_div.Divide(histos[1])
hist_div.SetLineColor(ROOT.kBlack)
hist_div.SetMarkerColor(ROOT.kBlack)
hist_div.GetYaxis().SetRangeUser(*y_ranges[1])

leg = ROOT.TLegend(*leg_position)

can = ROOT.TCanvas("c1","c1",700, 1000)

# TPad(name, title, xlow, ylow, xup, yup) - współrzędne od 0 do 1
# Górny pad (np. od 30% do 100% wysokości kanwy)
pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.3, 1.0, 1.0)
pad1.SetBottomMargin(0.02)  # Minimalny margines od dołu, żeby ukryć etykiety X
pad1.SetTopMargin(0.1)
pad1.Draw()

# Dolny pad (np. od 0% do 30% wysokości kanwy)
pad2 = ROOT.TPad("pad2", "pad2", 0.0, 0.0, 1.0, 0.3)
pad2.SetTopMargin(0.05)     # Minimalny margines od góry
pad2.SetBottomMargin(0.35)   # Większy margines na etykiety osi X dolnego wykresów
pad2.SetGrid()
pad2.Draw()

for i, hist in enumerate(histos):
    hist.SetLineColor(hist_color[i])
    hist.SetMarkerColor(hist_color[i])
    hist.GetYaxis().SetRangeUser(*y_ranges[0])
    hist.SetTitle(hist_title)
    leg.AddEntry(hist, leg_titles[i], "lp")

pad1.cd()
histos[0].Draw()
histos[1].Draw("SAME")

leg.Draw()

pad2.cd()

hist_div.SetTitle("") 
hist_div.GetYaxis().SetTitle("Ratio")
hist_div.GetYaxis().SetNdivisions(505) # Mniej kresek na osi Y, żeby się nie zlewały

# Kluczowe: skalowanie czcionek dla dolnego (mniejszego) padu
# Ponieważ pad ma 30% wysokości, czcionki muszą być powiększone współczynnikiem ~1/0.3
scale_factor = 1.0 / 0.45

hist_div.GetYaxis().SetLabelSize(0.04 * scale_factor)
hist_div.GetYaxis().SetTitleSize(0.05 * scale_factor)
hist_div.GetYaxis().SetTitleOffset(0.6) # Odsuwamy napis "Ratio", żeby nie nachodził na cyfry

hist_div.GetXaxis().SetLabelSize(0.04 * scale_factor)
hist_div.GetXaxis().SetTitleSize(0.05 * scale_factor)
hist_div.GetXaxis().SetTitleOffset(1.0)

# Upewnienie się, że zakresy osi X są identyczne na obu padach
hist_div.GetXaxis().SetRange(histos[0].GetXaxis().GetFirst(), histos[0].GetXaxis().GetLast())

hist_div.Draw("E1")

can.Draw()

name = f"{metric}_{var}_tag_vs_mc_truth_{ev_type}.pdf" if ev_type else f"{metric}_{var}_tag_vs_mc_truth.pdf"
can.SaveAs(f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/{name}")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/03e630a8/efficiency_purity_plots/corr_factor_Delta_t_tri_fit_tag_vs_mc_truth_MC.pdf has been created


In [11]:
import ROOT

cut_hash= "03e630a8"

var = "Delta_t_rec"
ev_type = ["data", "MC"]
histos_eff = []
hist_corr = None
hist_color = [ROOT.kRed, ROOT.kBlue]
leg_titles = ["Data", "MC"]
y_ranges = []
leg_position = (0.66, 0.82, 0.85, 0.9)
hist_title = ""

y_ranges = [(0, 1.1), (0.8, 1.2)]

eff_histograms = f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/eff_pur_corr_factor.root"

root_file = ROOT.TFile.Open(eff_histograms)

for et in ev_type:
    eff_dir = root_file.Get(f"efficiency_{et}")
    histos_eff.append(eff_dir.Get(f"efficiency_{var}_{et}"))

corr_dir = root_file.Get("corr_factor")
hist_corr = corr_dir.Get(f"corr_factor_{var}")

leg = ROOT.TLegend(*leg_position)

can = ROOT.TCanvas("c1","c1",700, 1000)

# TPad(name, title, xlow, ylow, xup, yup) - współrzędne od 0 do 1
# Górny pad (np. od 30% do 100% wysokości kanwy)
pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.3, 1.0, 1.0)
pad1.SetBottomMargin(0.02)  # Minimalny margines od dołu, żeby ukryć etykiety X
pad1.SetTopMargin(0.1)
pad1.Draw()

# Dolny pad (np. od 0% do 30% wysokości kanwy)
pad2 = ROOT.TPad("pad2", "pad2", 0.0, 0.0, 1.0, 0.3)
pad2.SetTopMargin(0.05)     # Minimalny margines od góry
pad2.SetBottomMargin(0.35)   # Większy margines na etykiety osi X dolnego wykresów
pad2.SetGrid()
pad2.Draw()

for i, hist in enumerate(histos_eff):
    hist.SetLineColor(hist_color[i])
    hist.SetMarkerColor(hist_color[i])
    hist.GetYaxis().SetRangeUser(*y_ranges[0])
    hist.SetTitle(hist_title)
    leg.AddEntry(hist, leg_titles[i], "lp")

pad1.cd()
histos_eff[0].Draw()
histos_eff[1].Draw("SAME")

leg.Draw()

pad2.cd()

hist_corr.GetYaxis().SetTitle("Data / MC")
hist_corr.SetLineColor(ROOT.kBlack)
hist_corr.SetMarkerColor(ROOT.kBlack)
hist_corr.GetYaxis().SetRangeUser(*y_ranges[1])

hist_corr.SetTitle("") 
hist_corr.GetYaxis().SetNdivisions(505) # Mniej kresek na osi Y, żeby się nie zlewały

# Kluczowe: skalowanie czcionek dla dolnego (mniejszego) padu
# Ponieważ pad ma 30% wysokości, czcionki muszą być powiększone współczynnikiem ~1/0.3
scale_factor = 1.0 / 0.45

hist_corr.GetYaxis().SetLabelSize(0.04 * scale_factor)
hist_corr.GetYaxis().SetTitleSize(0.05 * scale_factor)
hist_corr.GetYaxis().SetTitleOffset(0.6) # Odsuwamy napis "Ratio", żeby nie nachodził na cyfry

hist_corr.GetXaxis().SetLabelSize(0.04 * scale_factor)
hist_corr.GetXaxis().SetTitleSize(0.05 * scale_factor)
hist_corr.GetXaxis().SetTitleOffset(1.0)

# Upewnienie się, że zakresy osi X są identyczne na obu padach
hist_corr.GetXaxis().SetRange(histos_eff[0].GetXaxis().GetFirst(), histos_eff[0].GetXaxis().GetLast())

hist_corr.Draw("E1")

can.Draw()

name = f"{var}_corr_factor_final.pdf"
can.SaveAs(f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/{name}")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file results/ks_pm_control_sample/c97f42f6a601db1c64b593e259e04dec/img/03e630a8/efficiency_purity_plots/Delta_t_rec_corr_factor_final.pdf has been created
